# WSER Historical Results — Scraper & Exploration
Scrapes finisher data from wser.org (1974–2025), writes to CSV, loads into DuckDB for exploration.

**Schema:** `year, place, finish_time, first_name, last_name, gender, age, state`

In [ ]:
# Install dependencies if needed
# !pip install requests beautifulsoup4 pandas duckdb

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import duckdb
from pathlib import Path

SUMMARY_PATH = DATA_DIR / 'wser_year_summary.csv'

# Output paths
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / 'wser_results_raw.csv'
DB_PATH = DATA_DIR / 'wser.duckdb'

## 1. Scrape year-level summary (Starters, Finishers, Sub-24, Finish%)

In [6]:
def scrape_year_summary():
    """Scrape the index page for year-level summary stats."""
    url = 'https://www.wser.org/results/'
    resp = requests.get(url, timeout=15)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    rows = []
    table = soup.find('table')
    for tr in table.find_all('tr')[1:]:  # skip header
        cols = [td.get_text(strip=True) for td in tr.find_all('td')]
        if len(cols) >= 4:
            rows.append({
                'year': int(cols[0]),
                'starters': int(cols[1]) if cols[1].isdigit() else 0,
                'finishers': int(cols[2]) if cols[2].isdigit() else 0,
                'sub_24': int(cols[3]) if cols[3].isdigit() else 0,
            })
    return pd.DataFrame(rows)

df_summary = scrape_year_summary()
df_summary.to_csv(SUMMARY_PATH, index=False)
print(f"Year summary: {len(df_summary)} years")
df_summary.head(10)

Year summary: 52 years


,year,starters,finishers,sub_24
0,2025,369,285,92
1,2024,375,286,109
2,2023,379,328,110
3,2022,383,305,101
4,2021,315,208,57
5,2020,0,0,0
6,2019,369,319,130
7,2018,369,299,123
8,2017,369,248,76
9,2016,353,280,102


## 2. Scrape individual finisher results

In [31]:
def parse_time_to_minutes(time_str):
    """Convert HH:MM or H:MM:SS to total minutes. Returns None if unparseable."""
    try:
        parts = time_str.strip().split(':')
        if len(parts) >= 2:
            return int(parts[0]) * 60 + int(parts[1])
    except:
        return None


def scrape_year_results(year):
    """Scrape finisher results for a single year. Returns list of dicts."""
    url = f'https://www.wser.org/results/{year}-results/'
    try:
        resp = requests.get(url, timeout=15)
        if resp.status_code != 200:
            print(f"  {year}: HTTP {resp.status_code} — skipping")
            return []
    except Exception as e:
        print(f"  {year}: request error — {e}")
        return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table')
    if not table:
        print(f"  {year}: no table found")
        return []

    # Normalize headers to lowercase stripped strings
    headers = [th.get_text(strip=True).lower().strip() for th in table.find_all('th')]

    def find_col(row_dict, *keys):
        """Try multiple header aliases, return first match."""
        for k in keys:
            if k in row_dict:
                return row_dict[k]
        return None

    rows = []
    for tr in table.find_all('tr')[1:]:  # skip header row
        cols = [td.get_text(strip=True) for td in tr.find_all('td')]
        if not cols:
            continue

        row = dict(zip(headers, cols))

        # Handle both 'first'/'last' and 'first name'/'last name' headers
        # Also handles years with extra 'bib' and 'city' columns gracefully
        place    = find_col(row, 'place')
        time_str = find_col(row, 'time', 'finish time')
        first    = find_col(row, 'first name', 'first')
        last     = find_col(row, 'last name', 'last')
        gender   = find_col(row, 'gender')
        age      = find_col(row, 'age')
        state    = find_col(row, 'state/country', 'state')

        # Only keep rows with a valid finish time (excludes DNFs/DNSs)
        if not time_str or time_str.strip() == '':
            continue

        rows.append({
            'year': year,
            'place': place,
            'finish_time': time_str,
            'finish_minutes': parse_time_to_minutes(time_str) if time_str else None,
            'first_name': first,
            'last_name': last,
            'gender': gender,
            'age': age,
            'state': state,
        })

    return rows

In [32]:
# Years with no race (fire, COVID, etc.) — skip gracefully
SKIP_YEARS = {2008, 2020}

all_results = []
years = range(1974, 2026)

for year in years:
    if year in SKIP_YEARS:
        print(f"  {year}: skipped (no race)")
        continue
    
    results = scrape_year_results(year)
    all_results.extend(results)
    print(f"  {year}: {len(results)} finishers")
    time.sleep(0.75)  # be polite to wser.org

df = pd.DataFrame(all_results)
print(f"\nTotal rows: {len(df)}")
df.head()

  1974: 1 finishers
  1975: no table found
  1975: 0 finishers
  1976: 1 finishers
  1977: 3 finishers
  1978: 30 finishers
  1979: 96 finishers
  1980: 124 finishers
  1981: 146 finishers
  1982: 176 finishers
  1983: 196 finishers
  1984: 250 finishers
  1985: 163 finishers
  1986: 210 finishers
  1987: 183 finishers
  1988: 250 finishers
  1989: 246 finishers
  1990: 211 finishers
  1991: 242 finishers
  1992: 230 finishers
  1993: 209 finishers
  1994: 249 finishers
  1995: 198 finishers
  1996: 227 finishers
  1997: 257 finishers
  1998: 258 finishers
  1999: 216 finishers
  2000: 222 finishers
  2001: 267 finishers
  2002: 255 finishers
  2003: 272 finishers
  2004: 278 finishers
  2005: 318 finishers
  2006: 210 finishers
  2007: 273 finishers
  2008: skipped (no race)
  2009: 240 finishers
  2010: 328 finishers
  2011: 310 finishers
  2012: 316 finishers
  2013: 277 finishers
  2014: 296 finishers
  2015: 254 finishers
  2016: 280 finishers
  2017: 248 finishers
  2018: 299 fin

,year,place,finish_time,finish_minutes,first_name,last_name,gender,age,state
0,1974,1,23:42,1422,Ainsleigh,Gordy,M,26,CA
1,1976,1,24:30,1470,"Ken ""Cowman""",Shirk,M,NaN,
2,1977,1,22:57:00,1377,Andy,Gonzales,M,22,
3,1977,2,28:36:00,1716,Peter,Mattei,M,53,
4,1977,2,28:36:00,1716,Ralph,Paffenbarger,M,54,


In [33]:
# Light cleaning
df['place'] = pd.to_numeric(df['place'], errors='coerce')
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['gender'] = df['gender'].str.upper().str.strip()
df['state'] = df['state'].str.upper().str.strip()
df['first_name'] = df['first_name'].str.strip()
df['last_name'] = df['last_name'].str.strip()

# Save to CSV
df.to_csv(CSV_PATH, index=False)
print(f"Saved to {CSV_PATH}")
df.dtypes

Saved to data/wser_results_raw.csv


year                int64
place             float64
finish_time           str
finish_minutes      int64
first_name            str
last_name             str
gender                str
age               float64
state                 str
dtype: object

## 3. Load into DuckDB

In [8]:
con = duckdb.connect(str(DB_PATH))

# Main results table
con.execute("DROP TABLE IF EXISTS wser_results")
con.execute("""
    CREATE TABLE wser_results AS
    SELECT * FROM read_csv_auto(?)
""", [str(CSV_PATH)])

# Year summary table

con.execute("DROP TABLE IF EXISTS wser_year_summary")
con.execute("""
    CREATE TABLE wser_year_summary AS
    SELECT * FROM read_csv_auto(?)
""", [str(SUMMARY_PATH)])

print("Tables loaded:")
con.execute("SHOW TABLES").df()

Tables loaded:


,name
0,wser_results
1,wser_year_summary


## 4. Explore — sanity checks

In [9]:
# Row counts by year
con.execute("""
    SELECT year, COUNT(*) as finishers
    FROM wser_results
    GROUP BY year
    ORDER BY year
""").df()

,year,finishers
0,1974,1
1,1976,1
2,1977,3
3,1978,30
4,1979,96
5,1980,124
6,1981,146
7,1982,176
8,1983,196
9,1984,250


In [10]:
# Gender breakdown over time
con.execute("""
    SELECT year,
           COUNT(*) AS total,
           SUM(CASE WHEN gender = 'M' THEN 1 ELSE 0 END) AS male,
           SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) AS female,
           ROUND(100.0 * SUM(CASE WHEN gender = 'F' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_female
    FROM wser_results
    GROUP BY year
    ORDER BY year
""").df()

,year,total,male,female,pct_female
0,1974,1,1.0,0.0,0.0
1,1976,1,1.0,0.0,0.0
2,1977,3,3.0,0.0,0.0
3,1978,30,29.0,1.0,3.3
4,1979,96,90.0,6.0,6.3
5,1980,124,109.0,15.0,12.1
6,1981,146,127.0,19.0,13.0
7,1982,176,159.0,17.0,9.7
8,1983,196,175.0,21.0,10.7
9,1984,250,219.0,31.0,12.4


In [46]:
# Fastest finishes of all time
con.execute("""
    SELECT year, first_name, last_name, gender, age, finish_time, finish_minutes
    FROM wser_results
    WHERE finish_minutes IS NOT NULL
        AND gender = 'F'
    ORDER BY finish_minutes ASC
    LIMIT 20
""").df()

,year,first_name,last_name,gender,age,finish_time,finish_minutes
0,2023,Courtney,Dauwalter,F,38.0,15:29:33,929
1,2024,Katie,Schide,F,32.0,15:46:57,946
2,2024,Fuzhao,Xiang,F,32.0,16:20:03,980
3,2025,Abby,Hall,F,34.0,16:37:16,997
4,2024,Eszter,Csillag,F,39.0,16:42:17,1002
5,2023,Katie,Schide,F,31.0,16:43:45,1003
6,2012,Ellie,Greenwood,F,33.0,16:47:19,1007
7,2025,Fuzhao,Xiang,F,33.0,16:47:09,1007
8,2024,Emily,Hawgood,F,29.0,16:48:43,1008
9,2024,Yngvild,Kaspersen,F,29.0,16:50:39,1010


In [38]:
# Age distribution of finishers
con.execute("""
    SELECT 
        CASE 
            WHEN age < 30 THEN 'Under 30'
            WHEN age < 40 THEN '30s'
            WHEN age < 50 THEN '40s'
            WHEN age < 60 THEN '50s'
            ELSE '60+'
        END AS age_group,
        COUNT(*) AS finishers,
        ROUND(AVG(finish_minutes), 1) AS avg_finish_minutes
    FROM wser_results
    WHERE age IS NOT NULL AND finish_minutes IS NOT NULL
    GROUP BY age_group
    ORDER BY age_group
""").df()

,age_group,finishers,avg_finish_minutes
0,30s,3544,1461.4
1,40s,4479,1560.0
2,50s,1952,1631.1
3,60+,317,1689.8
4,Under 30,740,1404.0


In [39]:
# Top states by total finishers
con.execute("""
    SELECT state, COUNT(*) AS total_finishes
    FROM wser_results
    WHERE state IS NOT NULL AND state != ''
    GROUP BY state
    ORDER BY total_finishes DESC
    LIMIT 15
""").df()

,state,total_finishes
0,CA,1988
1,CO,291
2,OR,284
3,WA,206
4,TX,165
5,AZ,143
6,VA,105
7,UT,103
8,NV,101
9,GBR,93


In [40]:
# Runners with multiple finishes (the legends)
con.execute("""
    SELECT first_name, last_name,
           COUNT(*) AS total_finishes,
           MIN(year) AS first_year,
           MAX(year) AS last_year,
           ROUND(AVG(finish_minutes), 0) AS avg_finish_minutes
    FROM wser_results
    GROUP BY first_name, last_name
    HAVING COUNT(*) >= 5
    ORDER BY total_finishes DESC
    LIMIT 20
""").df()

,first_name,last_name,total_finishes,first_year,last_year,avg_finish_minutes
0,Tim,Twietmeyer,25,1981,2006,1118.0
1,Dan,Williams,22,1984,2019,1329.0
2,Gordy,Ainsleigh,21,1978,2007,1536.0
3,Jim,Scott,20,1992,2013,1338.0
4,Scott,Mills,20,1982,2019,1349.0
5,Mike,Pelechaty,20,1982,2010,1331.0
6,Charles,Savage,18,1980,2012,1476.0
7,Bill,Finkbeiner,17,1983,2011,1329.0
8,David,Kim,16,1985,2005,1517.0
9,Charley,Jones,15,2006,2023,1708.0


In [41]:
# Sub-24 rate trend by decade
con.execute("""
    SELECT 
        (year // 10) * 10 AS decade,
        COUNT(*) AS total_finishers,
        SUM(CASE WHEN finish_minutes < 1440 THEN 1 ELSE 0 END) AS sub_24,
        ROUND(100.0 * SUM(CASE WHEN finish_minutes < 1440 THEN 1 ELSE 0 END) / COUNT(*), 1) AS sub_24_pct
    FROM wser_results
    WHERE finish_minutes IS NOT NULL
    GROUP BY decade
    ORDER BY decade
""").df()

,decade,total_finishers,sub_24,sub_24_pct
0,1970,131,84.0,64.1
1,1980,1944,983.0,50.6
2,1990,2297,740.0,32.2
3,2000,2335,761.0,32.6
4,2010,2927,1147.0,39.2
5,2020,1412,469.0,33.2


In [45]:
# Quick look at YOUR finish if you want to find yourself in the data
con.execute("""
    SELECT last_name
        , COUNT(*)
    FROM wser_results
    GROUP BY 1 
    ORDER BY 2 DESC
""").df()

,last_name,count_star()
0,Williams,60
1,Smith,60
2,Scott,59
3,Johnson,55
4,Jones,53
...,...,...
4944,Rollins,1
4945,Luschin,1
4946,Gorichanaz,1
4947,Hopper,1


## 5. Next steps
- [ ] Set up Snowflake free tier and load CSV there
- [ ] Set up dbt Core project pointing at Snowflake
- [ ] Build staging model (clean/type the raw data)
- [ ] Build mart models (finisher trends, age group analysis, multi-finish runners)
- [ ] Build Streamlit explorer on top